In [56]:
from pylib.preamble import *
enable_autoreload()

from pylib.fig_setup import PALETTE, STYLE, LW, SEC_PALETTE, style_ax, legend, country_handles

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Effect of road transport electrification on EU aggregate rate

**Scope:** `FC_TRA_ROAD_E` ("Vejtransport") covers all road transport — trucks, personal cars, buses, motorcycles. There is no further sub-split at this disaggregation level in `nrg_bal_c`. Private car fuel is counted under road transport, not under households (`FC_OTH_HH_E` captures heating, cooking, and appliances only).

**Analysis:** What is the aggregate effect on the EU electrification rate of raising road transport from its current level to 100%?

1. Today's electrification rate: road transport and aggregate
2. Road transport's share of total final energy consumption
3. Counterfactual gain: $\Delta E_{\text{agg}} = s_{\text{road}} \cdot (1 - e_{\text{road}})$

*`1. elektrificeringsrater`*

*electrification data*

In [57]:
# 1. elektrificeringsrater
df_s_ = pd.read_parquet("0_intermediate/df_s.parquet")
df_a_ = pd.read_parquet("0_intermediate/df_a.parquet")

df_s = df_s_[
    df_s_.nrg_bal.eq('FC_TRA_ROAD_E') &
    df_s_.year.eq(2023) &
    df_s_.geo.eq('EU27_2020')
][['label', 'value']]

df_a = df_a_[
    df_a_.year.eq(2023) &
    df_a_.geo.eq('EU27_2020')
][['label','value']]

*`2. energivægte`*

*energy weight data (total final consumption)*

In [58]:
# 2. energivægte
df_enw_ = fetch_energy_weights(
    sectors=['FC_TRA_ROAD_E', 'FC_E'],
    countries=['EU27_2020'],
)
df_enw = df_enw_[df_enw_.year.eq(2023)]

road_tj  = df_enw.loc[df_enw.nrg_bal.eq('FC_TRA_ROAD_E'), 'value'].values[0]
total_tj = df_enw.loc[df_enw.nrg_bal.eq('FC_E'),          'value'].values[0]
s_road   = road_tj / total_tj

print(f's_road = {s_road:.2%}')

s_road = 30.10%


*`3. kontrafaktisk beregning`*

*Beregner den hypotetiske stigning i EU's aggregerede elektrificeringsrate ved fuld elektrificering af vejtransport, med et energieffektivitetsforhold $\eta = 2$ (1 enhed el leverer samme service som 2 enheder fossil energi):*

$$\Delta E_\text{agg} = \frac{s_\text{road} \cdot (1 - e_\text{road}/100)}{\eta}$$

In [ ]:
# 3. kontrafaktisk beregning
e_road = df_s['value'].values[0]   # road electrification rate (%)
e_agg  = df_a['value'].values[0]   # aggregate electrification rate (%)
eta    = 2                          # efficiency ratio: 1 unit electricity = eta units fossil service

delta_pp = s_road * (1 - e_road / 100) / eta * 100
e_cf     = e_agg + delta_pp

print("Today (EU-27, 2023)")
print(f"  Aggregate electrification rate:        {e_agg:.1f}%")
print(f"  Road transport electrification rate:    {e_road:.1f}%")
print(f"  Road share of total final energy:      {s_road:.1%}")
print()
print(f"Counterfactual: road transport electrified to 100% (η = {eta})")
print(f"  Gain:                                 +{delta_pp:.1f}pp")
print(f"  New aggregate electrification rate:    {e_cf:.1f}%")

*`4. trucks in european countries`*